# Modelo V3 — seguimiento completo con `p1_label..p12_label`

Esta versión utiliza las etiquetas semánticas de cada separación (`BEAM_TOP_FLANGE`, `BEAM_WEB_CENTRAL`, `HAUNCH_FLANGE`, `UNUSED`, etc.) que ya contiene el nuevo CSV.

A diferencia de V2, no es necesario inferir la semántica de `p1..p12` mediante reglas geométricas. La red predice conjuntamente:

- `DrawingCode`, `Bolt` y `SlipCritical`.
- `g, Bp, Lp, tp, e, d`.
- La etiqueta de cada posición `p1_label..p12_label`.
- El valor de cada `p1..p12`, aplicando máscara cuando su etiqueta es `UNUSED`.
- Los rigidizadores `tws, ts, tbs, tst`.

El notebook no modifica V2. Cambia `CSV_PATH` para apuntar al archivo completo en tu ordenador.

In [ ]:
from pathlib import Path
import sys, json, pickle
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

MODEL_DIR = Path.cwd()  # ejecutar el notebook desde V3/model
if str(MODEL_DIR) not in sys.path: sys.path.insert(0, str(MODEL_DIR))
from config import *
from dataset import Preprocessor, ConnectionDataset
from model import ConnectionDesignNet
from losses import ConnectionDesignLoss
from training import fit, predict
from postprocess import decode
from evaluation import evaluate

SEED=42
np.random.seed(SEED); torch.manual_seed(SEED)
CSV_PATH = MODEL_DIR / 'data_pi_tag.csv'  # cambia esta ruta si el CSV está en otra carpeta
CSV_SEPARATOR = ','
CSV_DECIMAL = '.'
print('PyTorch:',torch.__version__,'| CUDA:',torch.cuda.is_available())

## 1. Carga, esquema y calidad de datos

In [ ]:
df = pd.read_csv(CSV_PATH, sep=CSV_SEPARATOR, decimal=CSV_DECIMAL)
df.columns = df.columns.str.strip()
print('Dimensiones:',df.shape)
display(df.head())
print('Columnas:',df.columns.tolist())

In [ ]:
missing_cols=sorted(set(REQUIRED_COLUMNS)-set(df.columns))
assert not missing_cols, f'Faltan columnas obligatorias: {missing_cols}'
print('Columnas obligatorias: OK')
quality=pd.DataFrame({'dtype':df.dtypes.astype(str),'nulos':df.isna().sum(),'pct_nulos':100*df.isna().mean(),'unicos':df.nunique(dropna=False)})
display(quality.sort_values(['pct_nulos','unicos'],ascending=[False,True]))
print('Filas duplicadas completas:',df.duplicated().sum())

In [ ]:
# Normalización mínima y validaciones. No se eliminan duplicados automáticamente.
for c in CATEGORICAL_FEATURES + CLASSIFICATION_TARGETS + P_LABEL_COLUMNS:
    df[c]=df[c].fillna('UNUSED' if c in P_LABEL_COLUMNS else 'UNKNOWN').astype(str).str.strip()
for c in NUMERIC_FEATURES + REGRESSION_ALWAYS + P_COLUMNS + STIFFENER_COLUMNS:
    df[c]=pd.to_numeric(df[c],errors='coerce')
assert not df[NUMERIC_FEATURES+REGRESSION_ALWAYS].isna().any().any(), 'Hay NaN en entradas/targets continuos obligatorios'
# Una posición UNUSED debe tener valor 0; se informa y se corrige en la copia de trabajo.
for p,l in zip(P_COLUMNS,P_LABEL_COLUMNS):
    bad=(df[l]=='UNUSED') & (df[p].fillna(0)!=0)
    if bad.any(): print(f'Aviso: {bad.sum()} filas con {l}=UNUSED y {p}!=0; se fija a 0')
    df.loc[df[l]=='UNUSED',p]=0
df[P_COLUMNS+STIFFENER_COLUMNS]=df[P_COLUMNS+STIFFENER_COLUMNS].fillna(0)
print('Validación básica completada')

## 2. Exploración de objetivos

In [ ]:
fig,axes=plt.subplots(1,3,figsize=(16,4))
for ax,c in zip(axes,['DrawingCode','Bolt','SlipCritical']):
    df[c].value_counts().plot.bar(ax=ax,title=c); ax.tick_params(axis='x',rotation=45)
plt.tight_layout();plt.show()
print('Etiquetas p globales:')
display(pd.Series(df[P_LABEL_COLUMNS].to_numpy().ravel()).value_counts().to_frame('N'))

In [ ]:
fig,axes=plt.subplots(3,4,figsize=(18,10))
for ax,c in zip(axes.flat,P_LABEL_COLUMNS):
    df[c].value_counts().plot.bar(ax=ax,title=c);ax.tick_params(axis='x',rotation=70,labelsize=7)
plt.tight_layout();plt.show()

In [ ]:
display(df[REGRESSION_ALWAYS+P_COLUMNS+STIFFENER_COLUMNS].describe().T)
print('Porcentaje activo:')
display(pd.Series({c:100*(df[c].fillna(0)>0).mean() for c in P_COLUMNS+STIFFENER_COLUMNS},name='% activo'))

## 3. Separación train / validación / test
Se estratifica por `DrawingCode` cuando todas las clases tienen suficientes muestras. El test queda aislado hasta la evaluación final.

In [ ]:
def safe_stratify(frame,col='DrawingCode'):
    vc=frame[col].value_counts(); return frame[col] if len(vc)>1 and vc.min()>=2 else None
train_val,test_df=train_test_split(df,test_size=.20,random_state=SEED,stratify=safe_stratify(df))
train_df,val_df=train_test_split(train_val,test_size=.1875,random_state=SEED,stratify=safe_stratify(train_val)) # 65/15/20
for name,x in [('train',train_df),('val',val_df),('test',test_df)]:
    print(name,len(x),x['DrawingCode'].value_counts(normalize=True).round(3).to_dict())
train_df=train_df.reset_index(drop=True);val_df=val_df.reset_index(drop=True);test_df=test_df.reset_index(drop=True)

## 4. Preprocesador y DataLoaders
Todos los encoders y escaladores se ajustan **solo con train**. Las categorías desconocidas de entrada se aceptan mediante un índice reservado.

In [ ]:
pre=Preprocessor().fit(train_df)
print('Cardinalidades:',pre.cat_cardinalities)
print('DrawingCode:',pre.drawing_encoder.classes_)
print('Etiquetas p:',pre.p_label_encoder.classes_)
train_ds=ConnectionDataset(train_df,pre);val_ds=ConnectionDataset(val_df,pre);test_ds=ConnectionDataset(test_df,pre)
train_loader=DataLoader(train_ds,batch_size=64,shuffle=True,drop_last=len(train_ds)>=64)
val_loader=DataLoader(val_ds,batch_size=256);test_loader=DataLoader(test_ds,batch_size=256)
batch=next(iter(train_loader))
for k,v in batch.items():print(f'{k:18} {tuple(v.shape)} {v.dtype}')

## 5. Construcción del modelo

In [ ]:
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model=ConnectionDesignNet(pre.cat_cardinalities,len(NUMERIC_FEATURES),pre.n_drawing_codes,pre.n_p_labels).to(device)
print('Device:',device,'| Parámetros:',f'{sum(x.numel() for x in model.parameters()):,}')
model

In [ ]:
# Comprobación de dimensiones antes de entrenar
with torch.no_grad():
    out=model(batch['x_cat'].to(device),batch['x_num'].to(device))
for k,v in out.items():print(f'{k:20} {tuple(v.shape)}')

## 6. Entrenamiento

In [ ]:
EPOCHS=150
LEARNING_RATE=0.002
criterion=ConnectionDesignLoss()
history=fit(model,train_loader,val_loader,criterion,device,epochs=EPOCHS,lr=LEARNING_RATE)
print('Mejor época:',history['best_epoch'])

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(14,4))
axes[0].plot(history['train_loss'],label='train');axes[0].plot(history['val_loss'],label='val');axes[0].axvline(history['best_epoch']-1,ls=':',c='k');axes[0].legend();axes[0].set_title('Loss total')
for key in history['val_parts'][0]: axes[1].plot([x[key] for x in history['val_parts']],label=key)
axes[1].legend();axes[1].set_title('Componentes de validación');plt.show()

## 7. Evaluación final

In [ ]:
outputs=predict(model,test_loader,device)
pred_df=decode(outputs,pre)
cls_metrics,label_metrics,reg_metrics,structure_metrics=evaluate(test_df,pred_df)
print('Clasificación:',cls_metrics)
print('Estructura:',structure_metrics)
display(label_metrics.to_frame())
display(reg_metrics.round(3))

In [ ]:
fig,axes=plt.subplots(1,3,figsize=(17,4))
for ax,c in zip(axes,['DrawingCode','Bolt','SlipCritical']):
    labs=sorted(set(test_df[c].astype(str))|set(pred_df[c].astype(str)))
    ConfusionMatrixDisplay.from_predictions(test_df[c].astype(str),pred_df[c].astype(str),labels=labs,ax=ax,colorbar=False);ax.set_title(c);ax.tick_params(axis='x',rotation=45)
plt.tight_layout();plt.show()

In [ ]:
fig,axes=plt.subplots(2,3,figsize=(14,8))
for ax,c in zip(axes.flat,REGRESSION_ALWAYS):
    r=test_df[c].to_numpy();p=pred_df[c].to_numpy();ax.scatter(r,p,s=10,alpha=.3);lo=min(r.min(),p.min());hi=max(r.max(),p.max());ax.plot([lo,hi],[lo,hi],'r--');ax.set(title=c,xlabel='real',ylabel='predicho')
plt.tight_layout();plt.show()

In [ ]:
fig,axes=plt.subplots(3,4,figsize=(18,12))
for ax,c in zip(axes.flat,P_COLUMNS):
    m=test_df[c].fillna(0).to_numpy()>0;r=test_df.loc[m,c].to_numpy();p=pred_df.loc[m,c].to_numpy()
    if not m.any():ax.axis('off');continue
    ax.scatter(r,p,s=8,alpha=.3);lo=min(r.min(),p.min());hi=max(r.max(),p.max());ax.plot([lo,hi],[lo,hi],'r--');ax.set_title(f'{c} (N={m.sum()})')
plt.tight_layout();plt.show()

## 8. Inspección de casos y errores

In [ ]:
def compare_row(i):
    cols=['DrawingCode','Bolt','SlipCritical']+REGRESSION_ALWAYS
    table=pd.DataFrame({'real':test_df.loc[i,cols].astype(object),'predicho':pred_df.loc[i,cols].astype(object)})
    ptab=pd.DataFrame({'p_real':test_df.loc[i,P_COLUMNS].values,'label_real':test_df.loc[i,P_LABEL_COLUMNS].values,'p_pred':pred_df.loc[i,P_COLUMNS].values,'label_pred':pred_df.loc[i,P_LABEL_COLUMNS].values},index=P_COLUMNS)
    display(table);display(ptab)
compare_row(0)

In [ ]:
# Peores casos por suma de error de p activa + penalización por etiquetas distintas
real_p=test_df[P_COLUMNS].fillna(0).to_numpy(float); pred_p=pred_df[P_COLUMNS].to_numpy(float)
label_bad=(test_df[P_LABEL_COLUMNS].astype(str).to_numpy()!=pred_df[P_LABEL_COLUMNS].astype(str).to_numpy()).sum(1)
score=np.abs(real_p-pred_p).sum(1)+50*label_bad
worst=np.argsort(score)[::-1][:10]
display(pd.DataFrame({'indice_test':worst,'score':score[worst],'labels_incorrectas':label_bad[worst]}))
compare_row(int(worst[0]))

## 9. Guardado del bundle

In [ ]:
OUTPUT_DIR=MODEL_DIR/'artifacts';OUTPUT_DIR.mkdir(exist_ok=True)
torch.save(model.state_dict(),OUTPUT_DIR/'model.pt')
with open(OUTPUT_DIR/'preprocessor.pkl','wb') as f:pickle.dump(pre,f)
metadata={'version':'V3-p-labels','epochs':EPOCHS,'learning_rate':LEARNING_RATE,'best_epoch':history['best_epoch'],'n_train':len(train_df),'n_val':len(val_df),'n_test':len(test_df),'classification':cls_metrics,'structure':structure_metrics,'features':CATEGORICAL_FEATURES+NUMERIC_FEATURES}
(OUTPUT_DIR/'metadata.json').write_text(json.dumps(metadata,indent=2,default=float),encoding='utf-8')
print('Guardado en',OUTPUT_DIR.resolve())

## 10. Inferencia sobre filas nuevas
Las filas nuevas deben contener las columnas de entrada de `CATEGORICAL_FEATURES + NUMERIC_FEATURES`; no necesitan objetivos.

In [ ]:
@torch.no_grad()
def predict_dataframe(input_df):
    xcat,xnum=pre.transform_features(input_df)
    model.eval()
    raw=model(torch.as_tensor(xcat).to(device),torch.as_tensor(xnum,dtype=torch.float32).to(device))
    raw={k:v.cpu().numpy() for k,v in raw.items()}
    return decode(raw,pre)
example_prediction=predict_dataframe(test_df.head(5))
display(pd.concat([test_df[['BEAM','COLUMN']].head(5).reset_index(drop=True),example_prediction],axis=1))